# Closed loop in CARLA on Kaggle: perception -> decision -> control

Runs `ros2 launch perception_ros carla.launch.py` for each scenario in `configs/scenarios/`: CARLA 0.9.15
(headless) + `carla_bridge_node` + `perception_node` (the whole pipeline, on the GPU) + `decision_node`
(emergency brake). Each run writes `summary.json` (collision, first brake, stopping gap, ...), `ticks.csv` and
an overlay video, packed into one `.tar` under Output.

Kaggle's GPU is compute-only, so CARLA renders on the CPU (lavapipe) at a few seconds per frame, while the
perception models use the GPU. CARLA runs in synchronous mode and the bridge waits for the stack's command
every frame, so slow rendering only costs wall-clock time: the simulated 20 Hz loop is the same as a fast run.

ROS2 Humble comes from RoboStack (conda-forge), so ROS2, the CARLA client and PyTorch share one Python 3.10
environment, whatever Ubuntu version the Kaggle image runs.

**One-time setup:** two private Kaggle Datasets, attached with *Add Input*:
- `perception-stack-src`: `perception-stack-src.zip` (as for notebook 03; rebuild it after code changes)
- `perception-stack-weights`: `yolo11s_bdd100k_random.pt`, `twinlitenetplus_medium.ts`,
  `depth_anything_v2_small_relative_686x518.ts` (from WSL `weights/`)

**Each run:** Accelerator **GPU T4**, Internet **On**, edit the settings cell, then
**Save Version -> Save & Run All (Commit)** and download `<name>.tar` from Output when it is done.
Start with `SMOKE = True` (20 frames of one scenario, ~30-40 min including setup) before the full runs.

In [ ]:
# ---- settings ----
SMOKE = True  # True: 20 frames of the first scenario, to check the setup end to end
SCENARIOS = ["stopped_car", "pedestrian_crossing", "lead_brake"]  # configs/scenarios/<name>.yaml
BASELINE = ["stopped_car"]  # also run these with policy.type=noop: what happens without the stack
TOWN = "Town10HD_Opt"  # all scenarios must use this town (CARLA can't switch maps under CPU rendering)
NAME = "closed_loop_smoke" if SMOKE else "closed_loop"

In [ ]:
import glob
import json
import os
import shutil
import socket
import subprocess
import threading
import time


def sh(cmd, check=True, quiet=False, timeout=None):
    # Run a bash command, stream its output, return (returncode, last output lines).
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(
        ["bash", "-o", "pipefail", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    lines, start = [], time.time()
    try:
        for line in proc.stdout:
            lines.append(line)
            if not quiet:
                print(line, end="", flush=True)
            if timeout and time.time() - start > timeout:
                proc.kill()
                break
        proc.wait()
    except KeyboardInterrupt:  # Stop button: don't leave the command running in the background
        proc.kill()
        raise
    if check and proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}\n{''.join(lines[-40:])}")
    return proc.returncode, "".join(lines[-400:])


LOG = {"name": NAME, "smoke": SMOKE, "town": TOWN, "runs": {}}
WORK = "/tmp"
CARLA_DIR, ENV, REPO = f"{WORK}/carla", f"{WORK}/rosenv", f"{WORK}/repo"
OUT = "/kaggle/working"
RUNS = f"{WORK}/runs/{NAME}"
sh("free -g; echo cpus: $(nproc); grep PRETTY_NAME /etc/os-release; nvidia-smi -L; df -h /tmp | tail -1")

found = glob.glob("/kaggle/input/**/ros2_ws/src/perception_ros/package.xml", recursive=True)
assert found, "code not found: attach the perception-stack-src dataset with 'Add Input'"
SRC = found[0].split("/ros2_ws/")[0]
WEIGHTS = [
    "yolo11s_bdd100k_random.pt",
    "twinlitenetplus_medium.ts",
    "depth_anything_v2_small_relative_686x518.ts",
]
weight_paths = {w: glob.glob(f"/kaggle/input/**/{w}", recursive=True) for w in WEIGHTS}
missing = [w for w, p in weight_paths.items() if not p]
assert not missing, f"weights not found: {missing}; attach the perception-stack-weights dataset"

# Writable copy of the code (colcon builds inside it), weights linked in where the configs expect them
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.makedirs(f"{REPO}/weights", exist_ok=True)
for w, paths in weight_paths.items():
    os.symlink(paths[0], f"{REPO}/weights/{w}")
print("code:", SRC, "->", REPO)

## System packages and Vulkan (as notebook 03)

In [ ]:
apt = (
    "DEBIAN_FRONTEND=noninteractive timeout 600 apt-get install -y -qq --no-install-recommends "
    "-o DPkg::Lock::Timeout=120"
)
sh("timeout 300 apt-get update -qq", check=False, quiet=True)
sh(f"{apt} libvulkan1 vulkan-tools mesa-vulkan-drivers xdg-user-dirs bzip2 ffmpeg", quiet=True)
if sh(f"{apt} libomp5", check=False, quiet=True)[0] != 0:  # Ubuntu 24.04: libomp5-<llvm version>
    sh(f"{apt} libomp5-18", quiet=True)
for pkg in (
    "libsdl2-2.0-0",
    "libjpeg8",
    "libpng16-16t64",
    "fontconfig",
    "libtiff6",
    "libx11-6",
    "libxext6",
    "libgl1",
):
    sh(f"{apt} {pkg}", check=False, quiet=True)

# Kaggle's GPU has no graphics capability (compute only), so CARLA renders with lavapipe on the CPU
ICD = glob.glob("/usr/share/vulkan/icd.d/lvp_icd*.json")[0]
_, out = sh(f"VK_ICD_FILENAMES={ICD} vulkaninfo --summary 2>&1 | grep -E 'deviceName|ERROR'", check=False)
assert "llvmpipe" in out, "no working Vulkan driver"
LOG["renderer"] = "lavapipe (CPU)"

## CARLA server download and config (started after the ROS environment is built)

In [ ]:
CARLA_URLS = [
    "https://carla-releases.s3.us-east-005.backblazeb2.com/Linux/CARLA_0.9.15.tar.gz",
    "https://carla-releases.s3.eu-west-3.amazonaws.com/Linux/CARLA_0.9.15.tar.gz",
]
os.makedirs(CARLA_DIR, exist_ok=True)
if not os.path.exists(f"{CARLA_DIR}/CarlaUE4.sh"):
    for url in CARLA_URLS:
        start = time.time()
        if sh(f"curl -fsSL --retry 3 {url} | tar -xz -C {CARLA_DIR}", check=False)[0] == 0:
            LOG["carla_download_min"] = round((time.time() - start) / 60, 1)
            break
        print("failed:", url)
    else:
        raise RuntimeError("could not download CARLA")

# Start directly in the town: switching maps at runtime crashes under lavapipe, and the map argument on the
# command line is ignored, so set the default map in the engine config. CARLA refuses to run as root.
ini = f"{CARLA_DIR}/CarlaUE4/Config/DefaultEngine.ini"
map_ref = f"/Game/Carla/Maps/{TOWN}.{TOWN}"
sh(f"sed -i -E 's#^(GameDefaultMap|ServerDefaultMap)=.*#\\1={map_ref}#' {ini} && grep -E 'DefaultMap=' {ini}")
sh("id carla >/dev/null 2>&1 || useradd -m carla")
sh(f"chown -R carla:carla {CARLA_DIR}")
SERVER_LOG = f"{WORK}/carla_server.log"
start_cmd = (
    f"cd {CARLA_DIR} && VK_ICD_FILENAMES={ICD} exec ./CarlaUE4.sh -RenderOffScreen "
    f"-nosound -quality-level=Low -carla-rpc-port=2000 > {SERVER_LOG} 2>&1"
)


def start_server():
    # Kill any earlier server, start a new one in the background, return its start time
    sh("pkill -f CarlaUE4 || true; sleep 5", check=False, quiet=True)
    subprocess.Popen(
        ["su", "carla", "-s", "/bin/bash", "-c", start_cmd],
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    return time.time()


def port_open(port=2000):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("localhost", port)) == 0


def server_alive():
    return subprocess.run(["pgrep", "-f", "CarlaUE4-Linux"], capture_output=True).returncode == 0

## ROS2 Humble (RoboStack), CARLA client, PyTorch, the perception package

In [ ]:
start = time.time()
MM = f"{WORK}/bin/micromamba"
if not os.path.exists(MM):
    sh(f"curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba")
if not os.path.exists(f"{ENV}/bin/python"):
    # Python 3.10 because the carla 0.9.15 wheel stops there. RoboStack's newer Humble builds are 3.11+, so
    # this resolves to the older py310 builds on robostack-staging (they pin setuptools 61, fine for colcon).
    # Those builds predate two breaking releases: empy 4 (rosidl message generation fails on em.BUFFERED_OPT)
    # and GCC 13+ (the generated headers use uint32_t without including <cstdint>), so both are pinned.
    # No -q: on a solver error the message is in the last lines that sh() prints.
    sh(
        f"{MM} create -y -p {ENV} -c conda-forge -c robostack-staging python=3.10 'numpy<2' "
        "ros-humble-ros-base ros-humble-rmw-cyclonedds-cpp colcon-common-extensions "
        "gcc_linux-64=12 gxx_linux-64=12 cmake make pkg-config empy=3.3.4",
        quiet=True,
    )
PIP = f"{ENV}/bin/python -m pip install -q"
# The exported TorchScript models were traced with torch 2.5.1; same versions as the WSL venv
sh(f"{PIP} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124", quiet=True)
sh(
    f"{PIP} carla==0.9.15 ultralytics==8.4.183 'lap>=0.5.12' 'numpy<2' 'opencv-python-headless<4.11' "
    "omegaconf pandas pyarrow scipy pytest",
    quiet=True,
)
# Ultralytics pulls the full opencv-python (needs X libraries); keep only the headless one
sh(f"{ENV}/bin/python -m pip uninstall -y -q opencv-python || true", check=False, quiet=True)
sh(f"{PIP} --force-reinstall --no-deps 'opencv-python-headless<4.11'", quiet=True)
sh(f"{PIP} --no-deps -e {REPO}", quiet=True)
LOG["env_min"] = round((time.time() - start) / 60, 1)


def in_env(cmd, **kw):
    # Run cmd in the ROS environment, with the workspace sourced once it is built
    setup = (
        f"source {REPO}/ros2_ws/install/setup.bash && " if os.path.exists(f"{REPO}/ros2_ws/install") else ""
    )
    # pipefail: `colcon build | tail` must fail when the build does (sh()'s pipefail doesn't reach this shell)
    full = f"set -o pipefail; cd {REPO} && export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp && {setup}{cmd}"
    return sh(f"{MM} run -p {ENV} bash -c {json.dumps(full)}", **kw)


in_env(
    'python -c "import rclpy, carla, torch, ultralytics, perception; '
    "print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))\""
)
# Python3_EXECUTABLE: otherwise CMake finds Kaggle's system Python 3.13 and fails on its NumPy headers.
# On failure, print the errors from the failed packages' logs: the summary alone doesn't say what broke.
in_env(
    "cd ros2_ws && colcon build --symlink-install --event-handlers console_direct- "
    f"--cmake-args -DPython3_EXECUTABLE={ENV}/bin/python > build.log 2>&1 "
    "|| { tail -5 build.log; grep -h -i -B2 -A8 'error' log/latest_build/*/stderr.log | head -150; exit 1; } "
    "&& tail -5 build.log"
)
# ROS-side tests: message conversions, the nodes, the bridge's lockstep loop (with a fake simulator)
in_env("python -m pytest -q -p no:cacheprovider ros2_ws/src/perception_ros/test 2>&1 | tail -5")

In [ ]:
# Start the server only now, on an idle machine: lavapipe renders on the CPU, and while the environment
# was building on all cores its render thread fell more than 60 s behind and Unreal aborted ("GameThread timed
# out waiting for RenderThread", then a segfault). One retry in case a start still crashes.
for attempt in (1, 2):
    SERVER_STARTED = start_server()
    while (
        time.time() - SERVER_STARTED < 1800
        and not port_open()
        and (server_alive() or time.time() - SERVER_STARTED < 60)
    ):
        time.sleep(10)
    if port_open():
        break
    sh(f"tail -n 40 {SERVER_LOG}", check=False)
    print(f"CARLA server did not start (attempt {attempt})", flush=True)
else:
    raise RuntimeError("CARLA server did not start")
LOG["server_attempts"] = attempt
LOG["server_up_min"] = round((time.time() - SERVER_STARTED) / 60, 1)
with open(f"{WORK}/ping.py", "w") as f:
    f.write(
        "import carla\nc = carla.Client('localhost', 2000)\nc.set_timeout(900)\n"
        "print('server', c.get_server_version(), 'map', c.get_world().get_map().name)\n"
    )
_, out = in_env(f"python {WORK}/ping.py")
assert TOWN in out, f"server is not in {TOWN}"

## Scenarios

In [ ]:
import yaml

# GPU memory and RAM every 15 s: the GPU holds only the perception stack (CARLA renders on the CPU)
samples = []


def monitor():
    while not done.is_set():
        info = dict(line.split()[:2] for line in open("/proc/meminfo"))
        ram = (int(info["MemTotal:"]) - int(info["MemAvailable:"])) / 1e6
        # Kaggle has two T4s: one line per GPU, summed (the stack only uses the first)
        gpu = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
        ).stdout.split()
        samples.append(
            {
                "t_min": round((time.time() - t0) / 60, 1),
                "ram_gb": round(ram, 1),
                "gpu_mb": sum(map(int, gpu)),
            }
        )
        done.wait(15)


done, t0 = threading.Event(), time.time()
threading.Thread(target=monitor, daemon=True).start()

runs = [(s, "") for s in SCENARIOS] + [(s, "policy.type=noop") for s in BASELINE]
if SMOKE:
    runs = runs[:1]
os.makedirs(f"{WORK}/scenarios", exist_ok=True)
for scenario, overrides in runs:
    cfg = yaml.safe_load(open(f"{REPO}/configs/scenarios/{scenario}.yaml"))
    assert cfg["town"] == TOWN, f"{scenario} needs {cfg['town']}"
    if SMOKE:
        cfg["max_seconds"] = 1.0  # 20 frames
    path = f"{WORK}/scenarios/{scenario}.yaml"
    yaml.safe_dump(cfg, open(path, "w"))
    run = scenario + ("_no_policy" if "noop" in overrides else "")
    out_dir = f"{RUNS}/{run}"
    start = time.time()
    args = f"scenario:={path} out_dir:={out_dir} carla_timeout_s:=900 answer_timeout_s:=120"
    if overrides:  # ros2 launch rejects an empty argument ("malformed launch argument 'overrides:='")
        args += f" overrides:='{overrides}'"
    # Full launch output to launch.log in the run folder (packaged below); the cell shows it filtered
    os.makedirs(out_dir, exist_ok=True)
    rc, _ = in_env(
        f"ros2 launch perception_ros carla.launch.py {args} 2>&1 | tee {out_dir}/launch.log | "
        "grep -v -E 'debug_image|^\\s*$'",
        check=False,
        timeout=4 * 3600,
    )
    entry = {"rc": rc, "wall_min": round((time.time() - start) / 60, 1)}
    # ros2 launch exits 0 after a normal shutdown even when a node crashed: the summary is the real result
    if not os.path.exists(f"{out_dir}/summary.json"):
        entry["error"] = "no summary.json: the scenario did not finish (see launch.log)"
        sh(
            f"grep -E 'ERROR|Error|Exception|Traceback|died|did not appear' {out_dir}/launch.log | head -40",
            check=False,
        )
    else:
        entry["summary"] = json.load(open(f"{out_dir}/summary.json"))
        frames = sorted(glob.glob(f"{out_dir}/frames/*.jpg"))
        if frames:  # the overlay as a video at the simulated 20 fps
            sh(
                f"ffmpeg -loglevel error -y -framerate 20 -pattern_type glob -i '{out_dir}/frames/*.jpg' "
                f"-c:v libx264 -pix_fmt yuv420p {out_dir}/{run}.mp4",
                check=False,
            )
    LOG["runs"][run] = entry
    print(json.dumps({run: entry}, indent=2, default=str))
    if not server_alive():
        sh(f"tail -n 40 {SERVER_LOG}", check=False)
        print("CARLA server died: stopping")
        break
done.set()
LOG["memory"] = samples
LOG["peak_gpu_mb"] = max((s["gpu_mb"] for s in samples), default=None)
LOG["peak_ram_gb"] = max((s["ram_gb"] for s in samples), default=None)

## Package

In [ ]:
if os.path.exists(RUNS):
    sh(f"tar -cf {OUT}/{NAME}.tar -C {os.path.dirname(RUNS)} {NAME} && ls -lh {OUT}/{NAME}.tar")
with open(f"{OUT}/{NAME}_log.json", "w") as f:
    json.dump(LOG, f, indent=2, default=str)
sh("pkill -f CarlaUE4 || true", check=False, quiet=True)
for run, entry in LOG["runs"].items():
    s = entry.get("summary", {})
    brake = s.get("first_brake") or {}
    print(
        f"{run:28s} rc={entry['rc']} frames={s.get('frames')} collision={s.get('collision')} "
        f"braked={s.get('braked')} at gap {brake.get('gap_m')} m, stopped gap {s.get('stopped_gap_m')} m, "
        f"unanswered {s.get('unanswered_frames')}, {entry['wall_min']} min"
    )
print("peak GPU MB:", LOG["peak_gpu_mb"], " peak RAM GB:", LOG["peak_ram_gb"])